In [4]:

import torch
import torch.nn as nn
import torch.optim as optim

class NetBP(nn.Module):
    """
    4-Layer Artificial Neural Network described in Section 2 &amp; 6 (Figure 3).
    Architecture: 2 (Input) -&gt; 2 -&gt; 3 -&gt; 2 (Output) with Sigmoid activations.
    """
    def __init__(self):
        super(NetBP, self).__init__()
        self.fc2 = nn.Linear(2, 2)
        self.fc3 = nn.Linear(2, 3)
        self.fc4 = nn.Linear(3, 2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        a2 = self.sigmoid(self.fc2(x))
        a3 = self.sigmoid(self.fc3(a2))
        a4 = self.sigmoid(self.fc4(a3))
        return a4

def train_toy_network(num_iterations=100000, eta=0.05):
    # 10 Data points in R^2 (Section 2, Figure 1)
    x1 = torch.tensor([0.1, 0.3, 0.1, 0.6, 0.4, 0.6, 0.5, 0.9, 0.4, 0.7], dtype=torch.float32)
    x2 = torch.tensor([0.1, 0.4, 0.5, 0.9, 0.2, 0.3, 0.6, 0.2, 0.4, 0.6], dtype=torch.float32)
    X = torch.stack([x1, x2], dim=1) # Shape (10, 2)

    # Target labels: 5 Category A points [1], 5 Category B points [1]
    Y = torch.tensor([
        [1.0, 0.0], [1.0, 0.0], [1.0, 0.0], [1.0, 0.0], [1.0, 0.0],
        [0.0, 1.0], [0.0, 1.0], [0.0, 1.0], [0.0, 1.0], [0.0, 1.0]
    ], dtype=torch.float32)

    model = NetBP()
    
    # Scale weights by 0.5 * randn to match the paper's MATLAB initialization
    with torch.no_grad():
        for layer in [model.fc2, model.fc3, model.fc4]:
            layer.weight.copy_(0.5 * torch.randn_like(layer.weight))
            layer.bias.copy_(0.5 * torch.randn_like(layer.bias))

    optimizer = optim.SGD(model.parameters(), lr=eta)

    for step in range(1, num_iterations + 1):
        # Choose 1 random sample (sampling with replacement)
        k = torch.randint(0, 10, (1,)).item()
        
        optimizer.zero_grad()
        a4 = model(X[k:k+1])
        
        # Single sample quadratic loss: sum((y_k - a4_k)^2)
        loss = torch.sum((a4 - Y[k:k+1]) ** 2)
        loss.backward()
        optimizer.step()



In [10]:
class HighamCIFAR10CNN(nn.Module): 
    """ 5-Block Convolutional Neural Network from Section 7.4 (Figure 10). 
    """ 
    def __init__(self, use_dropout=True): 
        super(HighamCIFAR10CNN, self).__init__() 
# Block 1 
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=5, padding=2) 
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) 
        self.relu1 = nn.ReLU() 
        self.drop1 = nn.Dropout(p=0.15) if use_dropout else nn.Identity() 
# Block 2 
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=5, padding=2) 
        self.relu2 = nn.ReLU() 
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2) 
        self.drop2 = nn.Dropout(p=0.15) if use_dropout else nn.Identity() 
# Block 3 
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, padding=2) 
        self.relu3 = nn.ReLU() 
        self.pool3 = nn.AvgPool2d(kernel_size=2, stride=2) 
        self.drop3 = nn.Dropout(p=0.15) if use_dropout else nn.Identity() 
# Block 4 
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=4, padding=0) 
        self.relu4 = nn.ReLU() 
        self.drop4 = nn.Dropout(p=0.35) if use_dropout else nn.Identity() 
# Block 5: Fully Connected Layer 
        self.fc5 = nn.Linear(in_features=64, out_features=10)
        def forward(self, x): 
            x = self.drop1(self.relu1(self.pool1(self.conv1(x)))) 
            x = self.drop2(self.pool2(self.relu2(self.conv2(x)))) 
            x = self.drop3(self.pool3(self.relu3(self.conv3(x)))) 
            x = self.drop4(self.relu4(self.conv4(x))) 
            x = x.view(x.size(0), -1) # Flatten (N, 64, 1, 1) -&gt; (N, 64) 
            logits = self.fc5(x) 
            return logits 
def get_learning_rate(epoch): 
    if epoch <= 30: 
        return 0.05 
    elif epoch <= 40: 
        return 0.005 
    else: 
        return 0.0005
        